# Очистка данных о квартирах

Ноутбук содержит исследование качества объединённого датасета `public.flats_dataset`
и обоснование правил очистки для второго Airflow DAG.

Проверяются три обязательных класса проблем: **дубликаты, пропуски и выбросы**.
Финальная функция `clean_data()` в конце ноутбука предназначена для переноса в `plugins`.


## 1. Загрузка данных

Данные читаются из персональной PostgreSQL. Параметры подключения хранятся в `.env`
и не записываются в ноутбук.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

env_path = Path("../.env")
env = {}

for line in env_path.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    env[key] = value.strip().strip('"').strip("'")

required_vars = [
    "DB_DESTINATION_HOST",
    "DB_DESTINATION_PORT",
    "DB_DESTINATION_NAME",
    "DB_DESTINATION_USER",
    "DB_DESTINATION_PASSWORD",
]

missing_vars = [var for var in required_vars if not env.get(var)]
if missing_vars:
    raise ValueError(f"Missing environment variables: {missing_vars}")

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=env["DB_DESTINATION_USER"],
    password=env["DB_DESTINATION_PASSWORD"],
    host=env["DB_DESTINATION_HOST"],
    port=int(env["DB_DESTINATION_PORT"]),
    database=env["DB_DESTINATION_NAME"],
)

engine = create_engine(db_url)
df = pd.read_sql("SELECT * FROM public.flats_dataset", engine)
engine.dispose()

print("Shape:", df.shape)
display(df.head())


Shape: (141362, 18)


,flat_id,building_id,floor,is_apartment,kitchen_area,living_area,rooms,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,15700,18181,1,False,10.5,50.0,3,False,80.000000,9750000,2001,4,55.659512,37.777096,2.74,112,10,True
1,15701,7988,7,False,9.0,17.0,1,False,35.000000,13000000,1968,6,55.738037,37.677292,2.70,84,12,True
2,15702,165,5,False,0.0,0.0,5,False,240.699997,185000000,1912,1,55.754547,37.595322,4.00,15,7,True
3,15703,9859,4,False,6.0,0.0,3,False,60.000000,13950000,1972,4,55.606293,37.618774,2.64,717,9,True
4,15704,21036,5,False,8.6,18.0,1,False,39.500000,7500000,2008,4,55.681255,37.926819,3.00,403,17,True


## 2. Общая проверка качества

Сначала проверяем типы, описательные статистики, явные дубликаты и пропуски.
В исходном датасете явных дублей, дублей `flat_id` и пропусков обнаружено не было.


In [2]:
df.info()
display(df.describe(include="all").T)

print("Full duplicates:", df.duplicated().sum())
print("Duplicated flat_id:", df["flat_id"].duplicated().sum())

missing_report = pd.DataFrame({
    "missing_cnt": df.isna().sum(),
    "missing_share": df.isna().mean(),
}).sort_values("missing_cnt", ascending=False)

display(missing_report)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141362 entries, 0 to 141361
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   flat_id            141362 non-null  int64  
 1   building_id        141362 non-null  int64  
 2   floor              141362 non-null  int64  
 3   is_apartment       141362 non-null  bool   
 4   kitchen_area       141362 non-null  float64
 5   living_area        141362 non-null  float64
 6   rooms              141362 non-null  int64  
 7   studio             141362 non-null  bool   
 8   total_area         141362 non-null  float64
 9   price              141362 non-null  int64  
 10  build_year         141362 non-null  int64  
 11  building_type_int  141362 non-null  int64  
 12  latitude           141362 non-null  float64
 13  longitude          141362 non-null  float64
 14  ceiling_height     141362 non-null  float64
 15  flats_count        141362 non-null  int64  
 16  fl

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
flat_id,141362.0,NaN,NaN,NaN,70680.5,40807.838714,0.0,35340.25,70680.5,106020.75,141361.0
building_id,141362.0,NaN,NaN,NaN,14053.665235,6988.831066,1.0,8535.25,14332.0,20475.0,24620.0
floor,141362.0,NaN,NaN,NaN,7.467346,5.717144,1.0,3.0,6.0,10.0,56.0
is_apartment,141362,2,False,139990,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kitchen_area,141362.0,NaN,NaN,NaN,9.001579,5.264076,0.0,6.1,8.8,10.2,203.0
living_area,141362.0,NaN,NaN,NaN,31.056948,23.96864,0.0,19.0,29.4,41.400002,700.0
rooms,141362.0,NaN,NaN,NaN,2.129476,0.99434,1.0,1.0,2.0,3.0,20.0
studio,141362,1,False,141362,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_area,141362.0,NaN,NaN,NaN,62.374644,40.295864,11.0,39.299999,53.0,72.0,960.299988
price,141362.0,NaN,NaN,NaN,19441619.951904,66269544.299228,11.0,8900000.0,11850000.0,16950000.0,9873737728.0


Full duplicates: 0
Duplicated flat_id: 0


,missing_cnt,missing_share
flat_id,0,0.0
building_id,0,0.0
floor,0,0.0
is_apartment,0,0.0
kitchen_area,0,0.0
living_area,0,0.0
rooms,0,0.0
studio,0,0.0
total_area,0,0.0
price,0,0.0


## 3. Неявные дубликаты

Помимо полных дублей проверяем строки, где отличаются только `flat_id`, а все характеристики
квартиры и дома совпадают. Такие записи считаем неявными дублями и оставляем одну запись
из каждой группы.

В исходных данных найдено 8 309 таких групп; при `keep="first"` удаляется 9 116 строк.


In [3]:
implicit_duplicate_cols = [
    col for col in df.columns
    if col != "flat_id"
]

implicit_duplicate_mask = df.duplicated(
    subset=implicit_duplicate_cols,
    keep=False,
)

implicit_duplicate_groups = (
    df.loc[implicit_duplicate_mask]
      .groupby(implicit_duplicate_cols, dropna=False)["flat_id"]
      .agg(list)
      .reset_index(name="flat_ids")
)

implicit_duplicate_groups["duplicate_cnt"] = (
    implicit_duplicate_groups["flat_ids"].apply(len)
)

implicit_duplicate_groups = implicit_duplicate_groups.sort_values(
    "duplicate_cnt",
    ascending=False,
)

print("Rows in implicit duplicate groups:", implicit_duplicate_mask.sum())
print("Duplicate groups:", len(implicit_duplicate_groups))
print(
    "Rows to remove:",
    (implicit_duplicate_groups["duplicate_cnt"] - 1).sum(),
)

display(
    implicit_duplicate_groups[
        [
            "flat_ids",
            "duplicate_cnt",
            "building_id",
            "floor",
            "kitchen_area",
            "living_area",
            "rooms",
            "total_area",
            "price",
        ]
    ].head(10)
)


Rows in implicit duplicate groups: 17425
Duplicate groups: 8309
Rows to remove: 9116


,flat_ids,duplicate_cnt,building_id,floor,kitchen_area,living_area,rooms,total_area,price
822,"[140688, 140701, 140709, 140712, 140728, 14073...",10,3729,6,10.000000,30.000000,2,45.000000,30000000
7523,"[140690, 140708, 140720, 140726, 140730, 14073...",8,23510,19,10.000000,22.000000,2,56.700001,22000000
6999,"[19310, 19705, 36114, 100435, 100626, 100750, ...",8,22627,9,3.000000,0.000000,2,63.000000,10950000
3673,"[140702, 140713, 140718, 140723, 140729, 14073...",8,13890,15,10.700000,28.400000,2,49.599998,12800000
6830,"[100419, 100553, 100609, 100662, 100751, 10668...",7,22328,12,12.000000,0.000000,1,49.000000,7500000
7874,"[49622, 49623, 49624, 49625, 49626, 49627, 49628]",7,23996,3,33.599998,41.400002,3,119.500000,21629500
7885,"[49665, 49666, 49667, 49668, 49669, 49670, 49671]",7,23996,16,33.599998,41.400002,3,119.500000,23183000
3884,"[140703, 140714, 140724, 140736, 140746, 140749]",6,14501,8,9.700000,17.900000,1,34.000000,16490000
2584,"[113849, 113854, 113855, 113856, 113858, 113862]",6,10485,9,6.000000,38.000000,3,60.000000,12600000
7311,"[140696, 140705, 140710, 140716, 140721, 140731]",6,23156,2,11.900000,47.099998,3,82.599998,18000000


In [4]:
df_work = (
    df
    .drop_duplicates()
    .drop_duplicates(
        subset=implicit_duplicate_cols,
        keep="first",
    )
    .copy()
)

print("Rows before:", len(df))
print("Rows after deduplication:", len(df_work))
print("Removed:", len(df) - len(df_work))


Rows before: 141362
Rows after deduplication: 132246
Removed: 9116


## 4. Логические ошибки

Большие площади или большое число комнат сами по себе не считаются ошибками: для недвижимости
возможны редкие и дорогие объекты. Поэтому для площадей используем физические и логические
ограничения, а не автоматическое отсечение по IQR.

Главная обнаруженная проблема — `kitchen_area + living_area > total_area`.
После удаления неявных дублей этому условию соответствуют 148 строк.


In [5]:
def get_logical_issues_report(data):
    current_year = pd.Timestamp.now().year

    checks = {
        "price <= 0": data["price"] <= 0,
        "total_area <= 0": data["total_area"] <= 0,
        "kitchen_area < 0": data["kitchen_area"] < 0,
        "living_area < 0": data["living_area"] < 0,
        "rooms <= 0": data["rooms"] <= 0,
        "floor <= 0": data["floor"] <= 0,
        "floors_total <= 0": data["floors_total"] <= 0,
        "floor > floors_total": data["floor"] > data["floors_total"],
        "kitchen_area > total_area": data["kitchen_area"] > data["total_area"],
        "living_area > total_area": data["living_area"] > data["total_area"],
        "kitchen + living > total_area": (
            data["kitchen_area"] + data["living_area"] > data["total_area"]
        ),
        "build_year < 1800": data["build_year"] < 1800,
        "build_year > current year": data["build_year"] > current_year,
        "ceiling_height <= 0": data["ceiling_height"] <= 0,
        "invalid latitude": ~data["latitude"].between(-90, 90),
        "invalid longitude": ~data["longitude"].between(-180, 180),
    }

    return pd.DataFrame(
        {
            "issue": issue,
            "rows_cnt": mask.sum(),
            "rows_share": mask.mean(),
        }
        for issue, mask in checks.items()
    ).sort_values("rows_cnt", ascending=False)


logical_report = get_logical_issues_report(df_work)
display(logical_report)


,issue,rows_cnt,rows_share
10,kitchen + living > total_area,148,0.001119
8,kitchen_area > total_area,2,0.000015
0,price <= 0,0,0.000000
1,total_area <= 0,0,0.000000
4,rooms <= 0,0,0.000000
5,floor <= 0,0,0.000000
2,kitchen_area < 0,0,0.000000
3,living_area < 0,0,0.000000
7,floor > floors_total,0,0.000000
6,floors_total <= 0,0,0.000000


In [6]:
current_year = pd.Timestamp.now().year

logical_error_mask = (
    (df_work["price"] <= 0)
    | (df_work["total_area"] <= 0)
    | (df_work["kitchen_area"] < 0)
    | (df_work["living_area"] < 0)
    | (df_work["rooms"] <= 0)
    | (df_work["floor"] <= 0)
    | (df_work["floors_total"] <= 0)
    | (df_work["floor"] > df_work["floors_total"])
    | (df_work["kitchen_area"] > df_work["total_area"])
    | (df_work["living_area"] > df_work["total_area"])
    | (
        df_work["kitchen_area"]
        + df_work["living_area"]
        > df_work["total_area"]
    )
    | (df_work["build_year"] < 1800)
    | (df_work["build_year"] > current_year)
    | (df_work["ceiling_height"] <= 0)
    | (~df_work["latitude"].between(-90, 90))
    | (~df_work["longitude"].between(-180, 180))
)

print("Logical issue rows:", logical_error_mask.sum())

df_work = df_work.loc[~logical_error_mask].copy()
print("Rows after logical cleaning:", len(df_work))


Logical issue rows: 148
Rows after logical cleaning: 132098


## 5. Выбросы

Стандартный IQR по отдельным признакам оказался слишком агрессивным: распределения цены,
площади и ряда характеристик имеют естественный длинный правый хвост. Поэтому большие квартиры,
число комнат и этажность не удаляются только из-за редкости.

Для цены используется более сопоставимый показатель `price_per_sqm`. Его распределение
логарифмируется, после чего экстремальные значения определяются мягким правилом `3 × IQR`.
В текущем датасете оно выделяет 1 172 записи (около 0.89%).


In [7]:
df_work["price_per_sqm"] = (
    df_work["price"] / df_work["total_area"]
)

display(
    df_work["price_per_sqm"].describe(
        percentiles=[0.001, 0.01, 0.50, 0.99, 0.995, 0.999]
    )
)

print("Lowest price_per_sqm:")
display(
    df_work.nsmallest(5, "price_per_sqm")[
        ["flat_id", "price", "total_area", "price_per_sqm", "rooms"]
    ]
)

print("Highest price_per_sqm:")
display(
    df_work.nlargest(5, "price_per_sqm")[
        ["flat_id", "price", "total_area", "price_per_sqm", "rooms"]
    ]
)


count    1.320980e+05
mean     2.693403e+05
std      9.739053e+05
min      2.619048e-01
0.1%     1.032530e+03
1%       1.156250e+05
50%      2.333333e+05
99%      8.750000e+05
99.5%    1.109286e+06
99.9%    1.871962e+06
max      2.648649e+08
Name: price_per_sqm, dtype: float64

Lowest price_per_sqm:


,flat_id,price,total_area,price_per_sqm,rooms
44375,69633,11,42.0,0.261905,2
81948,107198,19,70.0,0.271429,2
79343,104573,85,59.0,1.440678,1
35119,60417,1500,36.0,41.666667,1
110705,135997,4000,95.0,42.105263,2


Highest price_per_sqm:


,flat_id,price,total_area,price_per_sqm,rooms
58553,83773,9799999488,37.0,2.648649e+08,2
20166,40855,9200000000,45.0,2.044444e+08,2
29737,50455,4686425088,103.0,4.549927e+07,3
60567,85787,9873737728,220.0,4.488063e+07,5
92821,118083,8147034112,208.0,3.916843e+07,4


In [8]:
log_price_per_sqm = np.log1p(df_work["price_per_sqm"])

q1 = log_price_per_sqm.quantile(0.25)
q3 = log_price_per_sqm.quantile(0.75)
iqr = q3 - q1

lower_log_bound = q1 - 3 * iqr
upper_log_bound = q3 + 3 * iqr

lower_price_bound = np.expm1(lower_log_bound)
upper_price_bound = np.expm1(upper_log_bound)

price_outlier_mask = (
    (log_price_per_sqm < lower_log_bound)
    | (log_price_per_sqm > upper_log_bound)
)

print("Lower price_per_sqm bound:", round(lower_price_bound, 2))
print("Upper price_per_sqm bound:", round(upper_price_bound, 2))
print("Price outliers:", price_outlier_mask.sum())
print("Share:", round(price_outlier_mask.mean() * 100, 4), "%")


Lower price_per_sqm bound: 56165.92
Upper price_per_sqm bound: 980637.84
Price outliers: 1172
Share: 0.8872 %


### Высота потолков

В распределении высоты потолков наблюдается единичное значение `27`, тогда как остальные
значения не превышают `8`. Это похоже на потерю десятичной точки (`27 → 2.7`).

Двузначное значение корректируется делением на 10 только тогда, когда результат попадает
в допустимый диапазон `[2, 8]`. После коррекции значения вне диапазона удаляются.


In [9]:
def fix_ceiling_height(value):
    if pd.isna(value):
        return value

    if 10 <= value < 100:
        corrected = value / 10
        if 2 <= corrected <= 8:
            return corrected

    return value


print(
    df_work.loc[df_work["ceiling_height"] > 4, "ceiling_height"]
    .value_counts()
    .sort_index()
)

df_work["ceiling_height"] = (
    df_work["ceiling_height"]
    .apply(fix_ceiling_height)
)

print(
    "Outside [2, 8] after correction:",
    (~df_work["ceiling_height"].between(2, 8)).sum(),
)


ceiling_height
4.05      3
4.10     12
4.15     43
4.19     36
4.50     13
4.60      4
5.00      5
5.20      1
6.00     14
7.00      4
8.00      6
27.00     1
Name: count, dtype: int64
Outside [2, 8] after correction: 0


## 6. Финальная функция очистки

Функция объединяет правила, полученные в ходе анализа:

1. удаление явных и неявных дублей;
2. удаление строк с пропусками;
3. удаление логически невозможных значений;
4. коррекция вероятной ошибки масштаба высоты потолка и проверка диапазона `[2, 8]`;
5. удаление экстремальных значений цены по логарифму цены за м² и правилу `3 × IQR`.

`price_per_sqm` используется только для контроля качества целевой переменной и не сохраняется
как признак модели.


In [10]:
def clean_data(data, price_iqr_factor=3.0):
    data = data.copy()

    # 1. Явные и неявные дубликаты
    data = data.drop_duplicates()

    implicit_duplicate_cols = [
        col for col in data.columns
        if col != "flat_id"
    ]

    data = data.drop_duplicates(
        subset=implicit_duplicate_cols,
        keep="first",
    )

    # 2. Пропуски
    data = data.dropna().copy()

    # 3. Логические ограничения
    current_year = pd.Timestamp.now().year

    logical_error_mask = (
        (data["price"] <= 0)
        | (data["total_area"] <= 0)
        | (data["kitchen_area"] < 0)
        | (data["living_area"] < 0)
        | (data["rooms"] <= 0)
        | (data["floor"] <= 0)
        | (data["floors_total"] <= 0)
        | (data["floor"] > data["floors_total"])
        | (data["kitchen_area"] > data["total_area"])
        | (data["living_area"] > data["total_area"])
        | (
            data["kitchen_area"]
            + data["living_area"]
            > data["total_area"]
        )
        | (data["build_year"] < 1800)
        | (data["build_year"] > current_year)
        | (data["ceiling_height"] <= 0)
        | (~data["latitude"].between(-90, 90))
        | (~data["longitude"].between(-180, 180))
    )

    data = data.loc[~logical_error_mask].copy()

    # 4. Высота потолков
    data["ceiling_height"] = (
        data["ceiling_height"]
        .apply(fix_ceiling_height)
    )

    data = data.loc[
        data["ceiling_height"].between(2, 8)
    ].copy()

    # 5. Экстремальные значения цены через цену за м²
    price_per_sqm = data["price"] / data["total_area"]
    log_price_per_sqm = np.log1p(price_per_sqm)

    q1 = log_price_per_sqm.quantile(0.25)
    q3 = log_price_per_sqm.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - price_iqr_factor * iqr
    upper_bound = q3 + price_iqr_factor * iqr

    price_valid_mask = log_price_per_sqm.between(
        lower_bound,
        upper_bound,
    )

    data = data.loc[price_valid_mask].copy()

    return data.reset_index(drop=True)


## 7. Контроль результата

Повторно запускаем все проверки уже для результата `clean_data()`.


In [11]:
df_clean = clean_data(df)

implicit_duplicate_cols = [
    col for col in df_clean.columns
    if col != "flat_id"
]

print("Rows before:", len(df))
print("Rows after:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))
print("Missing values:", int(df_clean.isna().sum().sum()))
print("Full duplicates:", df_clean.duplicated().sum())
print(
    "Implicit duplicates:",
    df_clean.duplicated(
        subset=implicit_duplicate_cols
    ).sum(),
)
print(
    "Invalid ceiling heights:",
    (~df_clean["ceiling_height"].between(2, 8)).sum(),
)

display(
    get_logical_issues_report(df_clean)
)


Rows before: 141362
Rows after: 130926
Rows removed: 10436
Missing values: 0
Full duplicates: 0
Implicit duplicates: 0
Invalid ceiling heights: 0


,issue,rows_cnt,rows_share
0,price <= 0,0,0.0
1,total_area <= 0,0,0.0
2,kitchen_area < 0,0,0.0
3,living_area < 0,0,0.0
4,rooms <= 0,0,0.0
5,floor <= 0,0,0.0
6,floors_total <= 0,0,0.0
7,floor > floors_total,0,0.0
8,kitchen_area > total_area,0,0.0
9,living_area > total_area,0,0.0
